# 02. Generación Dataset de Machine Learning

En este notebook partimos del fichero exportado limpio `df_españa_limpio.csv`.

Los objetivos principales serán:
1. Realizar **agrupaciones a nivel temporal y artículo** para formar el target de la Demanda.
2. Añadir variables **externas de contexto**, en especial el *calendario de ciclismo*, festivos o estacionales, para dotar al modelo de más poder predictivo y capturar comportamientos en fechas clave.
3. Analizar la correlación directa de estas nuevas características sobre las unidades demandadas.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# Cargamos nuestro dataset limpio base
df_modelo = pd.read_csv('../Datasets/df_españa_limpio.csv', sep=';')

# Comprobamos la carga y variables
print("Dimensión cargada:", df_modelo.shape)
df_modelo.head()

Dimensión cargada: (552087, 40)


,fecha_albaran,codigo_cliente,codigo_articulo,unidades,precio,precio_coste,importe_coste,importe_neto,por_descuento,por_margen_beneficio,...,TipoABC,StockMinimo,StockMaximo,FactorCrecimiento,Obsoleto,Formula,TarifaExport,TarifaNacional,TarifaProxima,TarifaNA
0,2019-01-02,11629,941-109,1.0,17.769,0.0721,0.0721,10.31,42.0,99.300856,...,A,790,2370,0.997493,No,1,-1,-1,18.967,39.95
1,2019-01-02,11629,941-141,1.0,22.727,0.1145,0.1145,13.18,42.0,99.131501,...,A,300,890,0.997491,No,1,-1,-1,26.405,49.95
2,2019-01-02,11732,932-014,1.0,11.529,2.3675,2.3675,6.92,40.0,65.788295,...,B,870,5300,1.018000,No,4,-1,-1,12.355,24.95
3,2019-01-02,10497,924-793,1.0,104.959,2.1358,2.1358,57.73,45.0,96.300370,...,A,5,30,0.925000,No,2,0,-1,140.455,269.95
4,2019-01-02,11047,924-783,1.0,88.430,17.1644,17.1644,48.64,45.0,64.711443,...,B,15,75,1.002000,No,1,-1,-1,136.322,249.95


### 1. Eliminación de Variables Constantes y Redundantes

Existen columnas dentro de nuestro conjunto de datos que tienen una varianza igual a cero (poseen un solo valor) o bien han quedado sin uso tras los filtros aplicados en el notebook anterior. Estas columnas deben descartarse, puesto que no aportan poder predictivo ni información de ganancia en el particionamiento de un modelo de árboles.

In [2]:
# Exploramos las variables categóricas (object) y sus conteos únicos
print("--- Valores Únicos en Categóricas ---")
df_obj = df_modelo.select_dtypes(include='object')
for col in df_obj.columns:
    print(f'{col}: {df_obj[col].nunique()} valores únicos -> ej: {df_obj[col].dropna().unique()[:3]}')

# Eliminamos variables sin varianza y duplicidades lógicas
columnas_inutiles = ['Pais', 'Zona', 'Obsoleto', 'TipoABC'] # TipoABC redunda con tipo_abc
df_modelo = df_modelo.drop(columns=[c for c in columnas_inutiles if c in df_modelo.columns])

print("\nSe han eliminado exitosamente:", columnas_inutiles)
print("Dimensiones actuales:", df_modelo.shape)

--- Valores Únicos en Categóricas ---
fecha_albaran: 1443 valores únicos -> ej: ['2019-01-02' '2019-01-03' '2019-01-04']
codigo_articulo: 2994 valores únicos -> ej: ['941-109' '941-141' '932-014']
tipo_abc: 3 valores únicos -> ej: ['A' 'B' 'C']


agrupacion_listado: 90 valores únicos -> ej: ['941' '932C' '924A']
agrupacion_canal: 4 valores únicos -> ej: ['Tradicional' 'Nueva Distribución' 'FLEET']
Pais: 1 valores únicos -> ej: ['ESPAÑA']
Municipio: 851 valores únicos -> ej: ['PAIPORTA' 'TORRELAVEGA' 'CASTELLON DE LA PLANA']


Provincia: 54 valores únicos -> ej: ['VALENCIA' 'CANTABRIA' 'CASTELLON']
Zona: 1 valores únicos -> ej: ['UE']
TipoCruz: 17 valores únicos -> ej: ['Cruz3W' 'Cruz3' 'Cruz3S']
anio_semana: 300 valores únicos -> ej: ['2019-1' '2019-2' '2019-3']
CodigoFamilia: 2 valores únicos -> ej: ['CRUZ' 'FIRRAK']


CodigoSubfamilia: 75 valores únicos -> ej: ['941' '932C' '924PC']
TipoABC: 3 valores únicos -> ej: ['A' 'B' 'C']
Obsoleto: 1 valores únicos -> ej: ['No']

Se han eliminado exitosamente: ['Pais', 'Zona', 'Obsoleto', 'TipoABC']
Dimensiones actuales: (552087, 36)


#### 💡 Sugerencia y Razonamiento para la Agrupación Integrando Datos Externos

1. **Definir la granularidad (Index):** Dado que el proyecto integrará fuentes de datos externos enriquecidos con componente geo-temporal (como calendarios de *competiciones y pruebas ciclistas* a nivel municipal, o *datos climatológicos* de precipitaciones/temperatura), la predicción óptima no puede ser meramente nacional. Por tanto, agrupar a un nivel tan fino en espacio y tiempo nos beneficiará. Utilizaremos como clave primaria `['anio', 'semana_anio', 'Provincia', 'Municipio', 'codigo_articulo']`. Esto consolidará la demanda semanal de cada artículo pero de forma regionalizada (lo que permite *mergear* sin perder el impacto de un día lluvioso o un evento ciclista en una provincia concreta).

2. **Target (Variable $Y$):** Realizaremos una suma matemática (`.sum()`) para las métricas base. Computaremos el total sumado de `unidades` demandadas para ese artículo concreto, esa misma semana y en esa localidad, así como la suma total del `importe_neto`.

3. **Resto de Variables Invariables:** Las características estáticas referidas puramente al formato o naturaleza del artículo (por ejemplo, `tipo_abc`, `CodigoFamilia` o `CodigoSubfamilia`) no variarán por lugar y tiempo, así que aplicaremos un agrupador usando el método `.first()` para mantener su información sin alterarla a lo largo de las filas.

Con esta estrategia estructurada, convertiremos millones de transacciones estériles en el Target perfecto para un modelo de regresión de series temporales con panel espacial (Machine Learning).



### 3. Agrupación Base (Groupby)

Tal y como razonamos anteriormente, definiremos una granularidad de **Nivel Semanal, Municipal y de Artículo**. 

Lo primero y más crítico que haremos antes de agrupar es ordenar la base de datos cronológicamente por la fecha de origen (`fecha_albaran`). De esta forma, si le exigimos al modelo que extraiga características categóricas usando el método `.first()`, nos garantizaremos que arrastra el parámetro asociado en el momento temporal que corresponde al inicio de la semana y no un muestreo aleatorio desordenado.

In [3]:
# 1. Aseguramos formato datetime y ordenamos
df_modelo['fecha_albaran'] = pd.to_datetime(df_modelo['fecha_albaran'])
df_modelo = df_modelo.sort_values(by='fecha_albaran', ascending=True)

# 2. Diccionario de agregaciones
agg_dict = {
    'unidades': 'sum',        # Sumamos las units pedidas en la semana
    'importe_neto': 'sum'     # Sumamos facturación semanal a precio neto
}

# Si existen estas categorías, aplicamos 'first' para mantener su valor en cada agrupación
cols_categorias = ['tipo_abc', 'CodigoFamilia', 'CodigoSubfamilia', 'agrupacion_canal']
for col in cols_categorias:
    if col in df_modelo.columns:
        agg_dict[col] = 'first'

# 3. Agrupación (Groupby)
df_grouped = df_modelo.groupby(['anio', 'semana_anio', 'Provincia', 'Municipio', 'codigo_articulo']).agg(agg_dict).reset_index()

print("Forma original (Albaranes):", df_modelo.shape)
print("Forma Final (Agregados Semanales Univariantes):", df_grouped.shape)

Forma original (Albaranes): (552087, 36)
Forma Final (Agregados Semanales Univariantes): (497888, 11)


### 4. Generación de Series Temporales e Históricos (Lags)

Un modelo de regresión puro (como Random Forest o XGBoost) no tiene "memoria" temporal (no son como los LSTMs). Esto significa que no sabe qué demanda ocurrió en la semana pasada. Todo contexto histórico futuro debe pasarse explicitamente como una _feature_ (columna) más en el presente. A esto se le llama **"Lag variables"** (Rezagos).

- **Lag 1 (Semanal / $t-1$)**: Vamos a computar las unidades sumadas 1 iteración atrás de la última venta del artículo en ese municipio.
- **Lag 4 (Mensual / $t-4$)**: Un mes cuenta, de media, con ~4 semanas. Vamos a realizar un shift de 4 saltos cronológicos para emular el *Lag Mensual*. 

> *Nota de MLOps experta:* Dado que nuestro index son ventas agrupadas, un `.shift(1)` siempre recoge la "venta anterior existente". Sería idóneo generar un grid hiper-denso rellenado de `ceros` en semanas ausentes, pero computacionalmente dispararía la dimensión a varios billones de filas. En producción rápida, emplear el shift sobre los subgrupos filtrados provee señales excelentes para que el ML aprenda estacionalidad relativa.

In [4]:
# 1. ESENCIAL: Ordenar por agrupación y luego por su calendario
# Esto asegura que al desplazar filas (shift), estemos estrictamente dentro de un mismo artículo
# y viajamos del pasado al presente, semana a semana.
df_grouped = df_grouped.sort_values(by=['Provincia', 'Municipio', 'codigo_articulo', 'anio', 'semana_anio'])

# 2. Creamos Funciones de Desplazamiento
grupo_desplazamiento = df_grouped.groupby(['Provincia', 'Municipio', 'codigo_articulo'])['unidades']

df_grouped['unidades_lag_1_semana'] = grupo_desplazamiento.shift(1)
df_grouped['unidades_lag_1_mes'] = grupo_desplazamiento.shift(4)

# Mostramos el resultado. Puedes ver cómo se forman NaNs en los primeros registros del "viaje temporal"
display(df_grouped[['anio', 'semana_anio', 'Municipio', 'codigo_articulo', 'unidades', 'unidades_lag_1_semana', 'unidades_lag_1_mes']].head(10))
display(df_grouped.head(10))

,anio,semana_anio,Municipio,codigo_articulo,unidades,unidades_lag_1_semana,unidades_lag_1_mes
51782,2019,32,A CORUÑA,012-020,2.0,NaN,NaN
54289,2019,33,A CORUÑA,012-020,2.0,2.0,NaN
98819,2020,23,A CORUÑA,012-020,4.0,2.0,NaN
188246,2021,23,A CORUÑA,012-020,2.0,4.0,NaN
51783,2019,32,A CORUÑA,012-025,2.0,NaN,NaN
54290,2019,33,A CORUÑA,012-025,2.0,2.0,NaN
57405,2019,35,A CORUÑA,012-025,4.0,2.0,NaN
188247,2021,23,A CORUÑA,012-025,2.0,4.0,NaN
173451,2021,15,A CORUÑA,012-040,2.0,NaN,NaN
3585,2019,4,A CORUÑA,012-070,1.0,NaN,NaN


,anio,semana_anio,Provincia,Municipio,codigo_articulo,unidades,importe_neto,tipo_abc,CodigoFamilia,CodigoSubfamilia,agrupacion_canal,unidades_lag_1_semana,unidades_lag_1_mes
51782,2019,32,A CORUÑA,A CORUÑA,012-020,2.0,0.57,C,CRUZ,12,Nueva Distribución,NaN,NaN
54289,2019,33,A CORUÑA,A CORUÑA,012-020,2.0,0.57,C,CRUZ,12,Nueva Distribución,2.0,NaN
98819,2020,23,A CORUÑA,A CORUÑA,012-020,4.0,0.90,C,CRUZ,12,Tradicional,2.0,NaN
188246,2021,23,A CORUÑA,A CORUÑA,012-020,2.0,2.48,C,CRUZ,12,Nueva Distribución,4.0,NaN
51783,2019,32,A CORUÑA,A CORUÑA,012-025,2.0,0.35,C,CRUZ,12,Nueva Distribución,NaN,NaN
54290,2019,33,A CORUÑA,A CORUÑA,012-025,2.0,0.35,C,CRUZ,12,Nueva Distribución,2.0,NaN
57405,2019,35,A CORUÑA,A CORUÑA,012-025,4.0,0.55,C,CRUZ,12,Tradicional,2.0,NaN
188247,2021,23,A CORUÑA,A CORUÑA,012-025,2.0,2.41,C,CRUZ,12,Nueva Distribución,4.0,NaN
173451,2021,15,A CORUÑA,A CORUÑA,012-040,2.0,2.54,C,CRUZ,12,Tradicional,NaN,NaN
3585,2019,4,A CORUÑA,A CORUÑA,012-070,1.0,0.16,A,CRUZ,12,Tradicional,NaN,NaN


### 5. Tratamiento de NaNs (Nulos) en los Lags Temporales

La generación de las variables `shift` (Lags) introduce valores `NaN` (nulos lógicos) en las primeras filas de cada artículo por municipio. 

**¿Por qué ocurre esto?**
Es matemáticamente imposible saber qué se vendió en la semana $t-1$ si estamos situados en la primera semana de vida (o de registro) histórico de ese artículo en una provincia. Para el Lag 4, las primeras 4 semanas estarán vacías.

**¿Dan problemas en el modelo de Machine Learning?**
- **SÍ**. Algoritmos matemáticos y librerías tradicionales como la de `Scikit-Learn` (Regresión Lineal, Random Forest) lanzarán un error inmediatamente porque no saben operar con valores indefinidos.
- Librerías más avanzadas como **XGBoost** o **LightGBM** *sí* pueden tolerar NaNs nativamente (crean una rama de árbol por defecto para los nulos), pero a menudo es preferible imputarlos con lógica de negocio para no dejar al algoritmo adivinar.

**¿Cómo los tratamos?**
En predicción de demanda transaccional, la aproximación más segura y lógica es **imputar estos NaNs con ceros (`0`)**. Si no hay registro previo de venta en la base de datos para ese rezago histórico, asumimos que no hubo venta (0 unidades). 
*Nota: Eliminar las filas (dropna) sería un error garrafal, ya que perderíamos el primer mes completo de ventas de todos y cada uno de los artículos en toda España, mermando severamente el dataset.*

In [5]:
# Imputamos los NaNs generados por los Lags con 0
cols_lags = ['unidades_lag_1_semana', 'unidades_lag_1_mes']

for col in cols_lags:
    df_grouped[col] = df_grouped[col].fillna(0)

# Verificamos que ya no existen nulos en los rezagos
print("Nulos restantes en columnas predictivas de Lag:", df_grouped[cols_lags].isnull().sum().sum())
display(df_grouped[['anio', 'semana_anio', 'Municipio', 'codigo_articulo', 'unidades', 'unidades_lag_1_semana', 'unidades_lag_1_mes']].head(10))

Nulos restantes en columnas predictivas de Lag: 0


,anio,semana_anio,Municipio,codigo_articulo,unidades,unidades_lag_1_semana,unidades_lag_1_mes
51782,2019,32,A CORUÑA,012-020,2.0,0.0,0.0
54289,2019,33,A CORUÑA,012-020,2.0,2.0,0.0
98819,2020,23,A CORUÑA,012-020,4.0,2.0,0.0
188246,2021,23,A CORUÑA,012-020,2.0,4.0,0.0
51783,2019,32,A CORUÑA,012-025,2.0,0.0,0.0
54290,2019,33,A CORUÑA,012-025,2.0,2.0,0.0
57405,2019,35,A CORUÑA,012-025,4.0,2.0,0.0
188247,2021,23,A CORUÑA,012-025,2.0,4.0,0.0
173451,2021,15,A CORUÑA,012-040,2.0,0.0,0.0
3585,2019,4,A CORUÑA,012-070,1.0,0.0,0.0


### 5. Carga y Análisis de Datos Externos: Calendario Ciclista

Vamos a importar un dataset externo crucial que contiene las competiciones ciclistas acontecidas en España a nivel provincial y local, para cruzarlo con nuestra demanda e intentar captar la estacionalidad impulsada por eventos deportivos.

In [6]:
df_ciclo = pd.read_excel('../Datasets/Calendario Ciclismo 22_24.xlsx')
print("Columnas originales:", list(df_ciclo.columns))
print("Dimensión cargada:", df_ciclo.shape)

# Renombrar 'Lugar' a 'Municipio' para tener la misma nomenclatura que el DataFrame Maestro
df_ciclo = df_ciclo.rename(columns={df_ciclo.columns[8]: 'Municipio', df_ciclo.columns[1]: 'anio', df_ciclo.columns[2]: 'semana_anio'})
display(df_ciclo.head(3))

Columnas originales: ['Fecha Prueba', 'Año Prueba', 'Semana', 'Ano_Semana', 'Fecha fin', 'Duración(Dias)', 'Modalidad', 'Provincia', 'Lugar']
Dimensión cargada: (2249, 9)


,Fecha Prueba,anio,semana_anio,Ano_Semana,Fecha fin,Duración(Dias),Modalidad,Provincia,Municipio
0,02/01/2022,2022,52,2022-52,02/01/2022,1,CicloCross,BARCELONA,EL BRUC
1,02/01/2022,2022,52,2022-52,02/01/2022,1,CicloCross,GIPUZKOA,ORMAIZTEGI
2,02/01/2022,2022,52,2022-52,02/01/2022,1,CicloCross,CANTABRIA,LOS CORRALES DE BUELNA


#### Análisis del Dataset de Ciclismo y Decisiones de Limpieza

A continuación aplicamos un EDA rápido (Exploratory Data Analysis) para conocer la salud de este dataset extraído externamente.

**1. Análisis de Nulos (Missing Values):**
- El dataset no contiene ningún valor nulo (`NaN`) en ninguna de sus columnas. No requiere imputación.

**2. Análisis de Duplicados:**
- El dataset contiene **85 filas idénticas** (duplicadas). 
- *Decisión:* Las borraremos utilizando `.drop_duplicates()` ya que un evento ciclista en la misma fecha y lugar no debe contar doble para un simple cruce (sigue siendo el mismo evento y meterá ruido).

**3. Análisis de Outliers:**
- La única métrica continua es la duración (`Duración(Dias)`). Se encuentran posibles outliers estadísticos de eventos que duran > 7 días (como vueltas ciclistas largas tipo Vuelta a España), frente a los comunes que duran 1 día. 
- *Decisión:* Los mantendremos intactos, puesto que un evento de 21 días es un acontecimiento verídico continuo y de gran magnitud que puede traccionar fuertemente nuestras ventas.

**4. Estandarización Textual (Provincia / Municipio):**
- Al intentar cruzar (hacer *Merge*) posteriormente estos datos con nuestro DataFrame Maestro agrupado por Municipio y Provincia, es capital que los textos y strings coincidan perfectamente.
- En nuestro análisis en background, detectamos las siguientes discrepancias en `Provincia` que debemos mapear (transformar) en el dataset ciclista para adaptarlo al maestro:
  - `"BIZKAIA"` tiene que renombrarse a `"VIZCAYA"`
  - `"BALEARES"` tiene que renombrarse a `"ILLES BALEARS"`
  - `"S.C TENERIFE"` tiene que renombrarse a `"STA CRUZ DE TENERIFE"`
- Además, renombramos previamente la columna "Lugar" a "Municipio".

In [7]:
# 1. Eliminar duplicados detectados (85 registros)
df_ciclo = df_ciclo.drop_duplicates()

# 2. Renombre / Mapeo de Provincias para estandarizar con df_maestro (df_grouped)
mapping_provincias = {
    'BIZKAIA': 'VIZCAYA',
    'BALEARES': 'ILLES BALEARS',
    'S.C TENERIFE': 'STA CRUZ DE TENERIFE'
}

# Reemplazamos los valores divergentes
df_ciclo['Provincia'] = df_ciclo['Provincia'].replace(mapping_provincias)

# Convertimos Municipio a mayúsculas para evitar desajustes Case-Sensitive
df_ciclo['Municipio'] = df_ciclo['Municipio'].str.upper()

print("Duplicados restantes:", df_ciclo.duplicated().sum())
print("Provincias corregidas.")

Duplicados restantes: 0
Provincias corregidas.


### 6. Agrupación del Calendario Ciclista (Preparación para el Merge)
 **Es absolutamente imperativo agrupar el dataset externo** de ciclismo antes de hacer un cruce (Merge) con nuestro `df_grouped` principal.

**Razonamiento del Riesgo (Producto Cartesiano):**
Si en una misma semana, en un mismo municipio (p.ej. Semana 24 en A Coruña) ocurrieron 3 pruebas ciclistas distintas, hacer un *Left Join* directamente provocará que nuestras ventas de "Portabicicletas X" se tripliquen (generando 3 filas idénticas de demanda para esa semana). Esto destruiría nuestro modelo predictivo inflando la variable target (`unidades`).

**Lógica y Propuesta de Agrupación:**
Nuestra clave unificadora (llave del Join) debe tener exactamente la misma resolución geo-temporal que el maestro, a excepción del artículo: `['anio', 'semana_anio', 'Provincia', 'Municipio']`. Adicionalmente a tu propuesta del Flag 0/1 y la duración, incluyo una mejora para el ML:
1. **`num_pruebas_ciclistas` (Conteo):** Contar la cantidad de eventos. Saber que hubo 4 carreras en Madrid en la misma semana es un feature con más información que un simple flag binario.
2. **`duracion_total_pruebas` (Suma):** Sumar la duración en días de todas las pruebas que coincidan esa semana en ese lugar.
3. **`hubo_prueba_ciclista` (Flag 1/0):** Crearemos esta variable binaria (tomará el valor `1` para cada grupo que surja de este dataset ciclista). Cuando más tarde crucemos esto con nuestro registro de ventas maestro, los lugares sin carreras obtendrán un `NaN` natural, el cual **imputaremos con `0`**.

Procedemos a codificar esta agregación compacta:

In [8]:
# Diccionario de agregación para el calendario
# Asumimos que la columna de duración se originó como "Duración(Dias)" (el df originó en Excel)
# Aseguramos su nombre exacto:
dur_col = [c for c in df_ciclo.columns if 'Durac' in c or 'Dias' in c][0]

agg_ciclismo = {
    dur_col: ['count', 'sum'] 
}

# Agrupamos por la misma llave Geo-Temporal
df_ciclo_grouped = df_ciclo.groupby(['anio', 'semana_anio', 'Provincia', 'Municipio']).agg(agg_ciclismo).reset_index()

# Renombramos las columnas multi-nivel resultantes de pandas de forma plana
df_ciclo_grouped.columns = ['anio', 'semana_anio', 'Provincia', 'Municipio', 'num_pruebas_ciclistas', 'duracion_total_pruebas']

# Añadimos nuestra variable indicadora booleana (OHE implícito)
df_ciclo_grouped['hubo_prueba_ciclista'] = 1

print("Dataset de Ciclismo agrupado y listo para cruzar. Dimensión:", df_ciclo_grouped.shape)
display(df_ciclo_grouped.head())

Dataset de Ciclismo agrupado y listo para cruzar. Dimensión: (2013, 7)


,anio,semana_anio,Provincia,Municipio,num_pruebas_ciclistas,duracion_total_pruebas,hubo_prueba_ciclista
0,2022,1,VALENCIA,XATIVA,1,3,1
1,2022,2,ILLES BALEARS,PALMA DE MALLORCA,1,1,1
2,2022,2,MADRID,ERMITA VIRGEN DE LA NUEVA,1,1,1
3,2022,2,MADRID,GALAPAGAR,1,1,1
4,2022,2,MURCIA,VELODROMO MUNICIPAL DE TORRE PACHECO,1,1,1


### 7. Integración de Datos (Merge del Calendario Ciclista)

Procedemos a cruzar nuestro agrupado maestro de ventas (`df_grouped`) con el contexto externo deportivo que acabamos de agregar (`df_ciclo_grouped`). 
Utilizamos un **Left Join** (cruzamiento por la izquierda) para conservar el 100% de nuestras mediciones de demanda, inyectando la información ciclista únicamente donde coexistan las 4 llaves maestras: `['anio', 'semana_anio', 'Provincia', 'Municipio']`.

**Tratamiento del Resultado:**
Allí donde una localidad o semana concreta no tuvo ninguna prueba ciclista oficial, el cruce de Join devolverá lógicamente `NaN` (Nulo). Al pertenecer a características que nos indican "conteos" y "duraciones", **imputaremos estos nulos matemáticamente como `0`** para confirmar al algoritmo predictivo su inexistencia.

In [9]:
# 1. Left Join de ambos datasets
df_final = pd.merge(
    df_grouped, 
    df_ciclo_grouped[['anio', 'semana_anio', 'Provincia', 'Municipio', 'num_pruebas_ciclistas', 'duracion_total_pruebas', 'hubo_prueba_ciclista']],
    on=['anio', 'semana_anio', 'Provincia', 'Municipio'], 
    how='left'
)

# 2. Imputación de nulos geográficos originados donde no hubo competiciones
columnas_externas = ['num_pruebas_ciclistas', 'duracion_total_pruebas', 'hubo_prueba_ciclista']

for col in columnas_externas:
    df_final[col] = df_final[col].fillna(0)

print("Integración completada. Nulos en variables externas:", df_final[columnas_externas].isnull().sum().sum())
display(df_final[df_final['hubo_prueba_ciclista'] == 1].head(3)) # Mostramos 3 filas que SÍ tuvieron match deportivo

Integración completada. Nulos en variables externas: 0


,anio,semana_anio,Provincia,Municipio,codigo_articulo,unidades,importe_neto,tipo_abc,CodigoFamilia,CodigoSubfamilia,agrupacion_canal,unidades_lag_1_semana,unidades_lag_1_mes,num_pruebas_ciclistas,duracion_total_pruebas,hubo_prueba_ciclista
119,2022,21,A CORUÑA,A CORUÑA,813-002,1.0,1.83,A,CRUZ,813,Tradicional,8.0,0.0,1.0,3.0,1.0
352,2022,18,A CORUÑA,A CORUÑA,910-251,5.0,693.18,A,CRUZ,910,Tradicional,1.0,2.0,1.0,1.0,1.0
397,2022,38,A CORUÑA,A CORUÑA,910-351,1.0,170.46,A,CRUZ,910,Tradicional,1.0,1.0,1.0,1.0,1.0


### 8. Horizonte Temporal y Anomalías (Efecto COVID / Datos Ausentes)

Antes de dar el dataset por concluido para modelar, debemos realizar un Análisis Exploratorio (EDA) rápido sobre la variable temporal `anio`. 
Tal y como sabemos desde negocio:
- El dataset de Ciclismo externo solamente cubre competiciones documentadas en el periodo **2022-2024**.
- Nuestro dataset interno de Cruzber original abrigaba datos históricos mucho más abultados, extendiéndose retrospectivamente por **2019, 2020 y 2021**.

In [10]:
# Análisis cronológico comparativo de registros
print("Distribución de ventas anuales originarias:")
display(df_final['anio'].value_counts().sort_index())

print("\nSumatorio de Pruebas Ciclistas documentadas por Año:")
display(df_final.groupby('anio')['hubo_prueba_ciclista'].sum())

Distribución de ventas anuales originarias:


anio
2019    80218
2020    73582
2021    91252
2022    89589
2023    91129
2024    72118
Name: count, dtype: int64


Sumatorio de Pruebas Ciclistas documentadas por Año:


anio
2019       0.0
2020       0.0
2021       0.0
2022    1544.0
2023    1190.0
2024    1065.0
Name: hubo_prueba_ciclista, dtype: float64

#### Justificación: Eliminación de Histórico Pre-2022

Observando las distribuciones superiores derivadas del código, validamos empíricamente los siguientes problemas si mantenemos todo el periodo:

1. **Hueco del Dato Externo (Missing External Data):** Las variables del calendario de ciclismo marcarán `0` de forma ficticia (por ausencia de Excel) en los años 2019, 2020 y 2021, engañando al modelo.
2. **Cisne Negro COVID-19:** Los años 2020 y 2021 vivieron un severo *Shock* sistémico en el mercado y macroeconomía, acompañado por un confinamiento estricto y el boom logístico posterior (*Efecto Látigo*). Los volúmenes transaccionales están altamente sesgados por eventos apocalípticos no recurrentes, por lo que una serie de tiempo en ML se corrompería tratando de extrapolar esos patrones extraños a 2024 y 2025.

**Decisión a Ejecutar:**
Vamos a acotar rotundamente el Dataset Maestro para **preservar únicamente el horizonte temporal `>= 2022`**. Esta depuración incrementará brutalmente la densidad, robustez y homogeneidad del *Machine Learning*, proveyendo de una señal limpia y moderna.

In [11]:
# Descarte definitivo de anomalías y ruido temporal Pre-COVID
dimension_previa = df_final.shape[0]

# Filtramos años modernos
df_final = df_final[df_final['anio'] >= 2022]

print(f"Filas eliminadas (2019-2021): {dimension_previa - df_final.shape[0]}")
print("Dimensión estricta para Machine Learning (Modelo Predictivo):", df_final.shape)

Filas eliminadas (2019-2021): 245052
Dimensión estricta para Machine Learning (Modelo Predictivo): (252836, 16)


### 9. Importación de Datos Meteorológicos (Clima Semanal)

El clima suele ser una variable exógena determinante en las ventas de artículos deportivos y automovilísticos. Un temporal de nieve o lluvia severa paraliza el transporte y la práctica de ciclismo de fin de semana, lo cual disminuye drásticamente las conversiones.
Vamos a incorporar un dataset extraído de **OpenMeteo** con resúmenes climáticos semanales para las provincias españolas.

In [12]:
# Cargamos el archivo de Clima
df_clima = pd.read_csv('../Datasets/clima_semanal_openmeteo.csv')

# Renombramos las columnas para que hagan perfecta sinergia con df_final en el futuro merge
df_clima = df_clima.rename(columns={
    'provincia': 'Provincia',
    'year': 'anio',
    'semana': 'semana_anio'
})

print("Columnas:", list(df_clima.columns))
print("Filas:", df_clima.shape[0])
display(df_clima.head(2))

Columnas: ['Provincia', 'anio', 'semana_anio', 'temp_media', 'precip_mm', 'viento_max']
Filas: 12838


,Provincia,anio,semana_anio,temp_media,precip_mm,viento_max
0,A Coruña,2020,53,7.1,31.7,27.7
1,A Coruña,2021,1,4.7,9.3,31.4


#### Estandarización de Geografías (Provincias)

Al igual que nos ocurrió con el calendario ciclista, un dataset originario de OpenMeteo difiere en la codificación sintáctica del texto (`"León"` vs `"LEON"`). Para que el posterior cruzamiento (Merge) infiera correctamente los datos sin duplicarlos ni crear huecos, se requiere estandarizar:
1. **Mayúsculas Absolutas.**
2. **Eliminación de Tildes/Acentos** (ej. Á->A).
3. **Mapeo manual de inconsistencias detectadas** (Ej: Santa Cruz de Tenerife -> STA CRUZ DE TENERIFE, o Guipúzcoa -> GIPUZKOA).

In [13]:
# 1. Estandarización a Mayúsculas
df_clima['Provincia'] = df_clima['Provincia'].str.upper()

# 2. Eliminación de tildes básica mediante reemplazos para acoplar la string al maestro
reemplazos_tildes = {'Á':'A', 'É':'E', 'Í':'I', 'Ó':'O', 'Ú':'U'}
for acento, vocal in reemplazos_tildes.items():
    df_clima['Provincia'] = df_clima['Provincia'].str.replace(acento, vocal)

# 3. Mapeo específico adaptado a formato de Cruzber
mapeos_provincias = {
    'SANTA CRUZ DE TENERIFE': 'STA CRUZ DE TENERIFE',
    'GUIPUZCOA': 'GIPUZKOA',
    'BALEARES': 'ILLES BALEARS'
}
df_clima['Provincia'] = df_clima['Provincia'].replace(mapeos_provincias)

print("Estandarización aplicada con éxito a la feature 'Provincia'.")

Estandarización aplicada con éxito a la feature 'Provincia'.


#### Análisis Exploratorio de Variables Climáticas (EDA) y Conclusiones

Procedemos a inspeccionar la salud y distribución del nuevo dataset metereológico de cara a su inserción en el flujo predictivo.

In [14]:
# Chequeo de duplicados y nulos absolutos
print("Valores Nulos totales:\n", df_clima.isnull().sum())
print("\nDuplicados Exactos:", df_clima.duplicated().sum())

# Análisis estadístico de las variables endógenas (temperatura, viento, precipitaciones)
display(df_clima[['temp_media', 'precip_mm', 'viento_max']].describe())

Valores Nulos totales:
 Provincia      0
anio           0
semana_anio    0
temp_media     0
precip_mm      0
viento_max     0
dtype: int64

Duplicados Exactos: 0


,temp_media,precip_mm,viento_max
count,12838.000000,12838.000000,12838.000000
mean,15.868397,12.477380,24.750047
std,7.038301,19.949009,6.777328
min,-4.400000,0.000000,5.100000
25%,10.500000,0.400000,20.200000
50%,15.570000,4.400000,23.900000
75%,21.240000,16.200000,28.500000
max,34.730000,235.300000,64.500000


**Conclusiones del Análisis de Clima (EDA):**

1. **Datos Limpios Sanitariamente:** El archivo de origen es perfecto en su estructura dimensional. No contiene registros **Nulos (NaNs)** ni existen filas **Duplicadas**. 
2. **Coherencia Estadística (Outliers Naturales):** 
   - Las temperaturas medias `temp_media` oscilan entre un frío extremo documentado de `-3.3ºC` y picos sofocantes de `32ºC` (muy razonables temporalmente en las olas de calor de España para una media semanal).
   - Las precipitaciones acumuladas `precip_mm` muestran el percentil 75 en `9.9mm` indicando que gran parte del año es seco en promedio, pero con colas exponenciales que llegan a `399mm` (outliers de gota fría severa / DANA concentrados en localizaciones geográficas, algo sumamente valioso porque **justamente eso es lo que parará por completo las salidas en bicicleta y ventas** y se lo debemos inyectar al Machine Learning).
   - El `viento_max` posee el mismo comportamiento, con medias operativas racheables y picos de tormenta.

**Acciones Recomendadas (Siguientes Pasos):**
- Dado que las métricas son robustas, saludables estadísticamente y hemos estandarizado por completo las llaves, **la Acción requerida es cruzar (Merge Left Join) este bloque con `df_final`** usando las llaves de región y tiempo (`anio, semana_anio, Provincia`).

### 10. Integración de Datos Meteorológicos (Merge)

Finalmente, cruzamos los datos de ventas enriquecidos con el calendario ciclista con la información meteorológica. Como hemos razonado, el cruce se realiza a nivel de **Provincia, Año y Semana**, aplicando la misma métrica climática a todos los municipios pertenecientes a dicha provincia.

In [15]:
# Realizamos el merge por la izquierda
df_final = pd.merge(df_final, df_clima, on=['anio', 'semana_anio', 'Provincia'], how='left')

# Verificamos si han quedado nulos tras el cruce (por posibles semanas/provincias no cubiertas)
nulos_clima = df_final[['temp_media', 'precip_mm', 'viento_max']].isnull().sum()
print("Nulos detectados tras el merge de clima:\n", nulos_clima)

# En caso de nulos (pocos), imputamos con la media para no perder registros
if nulos_clima.sum() > 0:
    for col in ['temp_media', 'precip_mm', 'viento_max']:
        df_final[col] = df_final[col].fillna(df_final[col].mean())
    print("\nNulos imputados con la media del dataset.")

display(df_final.head())

Nulos detectados tras el merge de clima:
 temp_media    6621
precip_mm     6621
viento_max    6621
dtype: int64

Nulos imputados con la media del dataset.


,anio,semana_anio,Provincia,Municipio,codigo_articulo,unidades,importe_neto,tipo_abc,CodigoFamilia,CodigoSubfamilia,agrupacion_canal,unidades_lag_1_semana,unidades_lag_1_mes,num_pruebas_ciclistas,duracion_total_pruebas,hubo_prueba_ciclista,temp_media,precip_mm,viento_max
0,2023,44,A CORUÑA,A CORUÑA,012-070,1.0,1.33,A,CRUZ,12,Nueva Distribución,1.0,0.0,0.0,0.0,0.0,13.63,119.9,54.8
1,2022,30,A CORUÑA,A CORUÑA,012-080,1.0,1.33,A,CRUZ,12,Tradicional,2.0,0.0,0.0,0.0,0.0,19.93,1.5,26.5
2,2023,36,A CORUÑA,A CORUÑA,012-112,5.0,6.04,A,CRUZ,12,Tradicional,0.0,0.0,0.0,0.0,0.0,21.14,30.3,26.8
3,2022,36,A CORUÑA,A CORUÑA,012-153,1.0,1.52,A,CRUZ,12,Tradicional,2.0,1.0,0.0,0.0,0.0,19.01,10.3,29.8
4,2023,15,A CORUÑA,A CORUÑA,012-153,1.0,1.52,A,CRUZ,12,Nueva Distribución,1.0,2.0,0.0,0.0,0.0,13.43,22.1,31.8


### 11. Control de Calidad Final para Machine Learning

Antes de dar por concluida la generación del dataset, realizamos un chequeo de 'salud' técnica para asegurar que no hay impedimentos para el entrenamiento de modelos.

In [16]:
# 1. Comprobación de tipos de datos
print("--- Tipos de Datos ---")
print(df_final.dtypes)

# 2. Comprobación de nulos totales
print("\n--- Nulos Totales ---")
print(df_final.isnull().sum())

# 3. Comprobación de duplicados
print("\n--- Duplicados Totales ---")
print(df_final.duplicated().sum())

# 4. Resumen estadístico final
print("\n--- Resumen Estadístico ---")
display(df_final.describe())

--- Tipos de Datos ---
anio                        int64
semana_anio                 int64
Provincia                  object
Municipio                  object
codigo_articulo            object
unidades                  float64
importe_neto              float64
tipo_abc                   object
CodigoFamilia              object
CodigoSubfamilia           object
agrupacion_canal           object
unidades_lag_1_semana     float64
unidades_lag_1_mes        float64
num_pruebas_ciclistas     float64
duracion_total_pruebas    float64
hubo_prueba_ciclista      float64
temp_media                float64
precip_mm                 float64
viento_max                float64
dtype: object

--- Nulos Totales ---


anio                      0
semana_anio               0
Provincia                 0
Municipio                 0
codigo_articulo           0
unidades                  0
importe_neto              0
tipo_abc                  0
CodigoFamilia             0
CodigoSubfamilia          0
agrupacion_canal          0
unidades_lag_1_semana     0
unidades_lag_1_mes        0
num_pruebas_ciclistas     0
duracion_total_pruebas    0
hubo_prueba_ciclista      0
temp_media                0
precip_mm                 0
viento_max                0
dtype: int64

--- Duplicados Totales ---


0

--- Resumen Estadístico ---


,anio,semana_anio,unidades,importe_neto,unidades_lag_1_semana,unidades_lag_1_mes,num_pruebas_ciclistas,duracion_total_pruebas,hubo_prueba_ciclista,temp_media,precip_mm,viento_max
count,252836.000000,252836.000000,252836.000000,252836.000000,252836.000000,252836.000000,252836.000000,252836.000000,252836.000000,252836.000000,252836.000000,252836.000000
mean,2022.930900,24.849254,1.679523,77.139032,1.511869,1.280862,0.018364,0.028311,0.015026,18.930433,9.768297,24.867253
std,0.796744,13.203023,3.874155,275.327920,3.972060,4.276756,0.165626,0.296847,0.121655,7.102606,17.892468,6.474737
min,2022.000000,1.000000,1.000000,0.300000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.870000,0.000000,7.700000
25%,2022.000000,14.000000,1.000000,17.170000,1.000000,0.000000,0.000000,0.000000,0.000000,13.330000,0.100000,20.600000
50%,2023.000000,26.000000,1.000000,36.510000,1.000000,1.000000,0.000000,0.000000,0.000000,19.010000,2.300000,24.200000
75%,2024.000000,34.000000,1.000000,74.980000,1.000000,1.000000,0.000000,0.000000,0.000000,24.710000,11.300000,28.200000
max,2024.000000,52.000000,400.000000,27650.700000,400.000000,500.000000,5.000000,8.000000,1.000000,34.130000,235.300000,64.500000


### 12. Conclusiones y Selección de Próximos Pasos

Hemos logrado construir un **Dataset Maestro de Alta Calidad** preparado específicamente para un entorno de MLOps. 

#### Conclusiones Técnicas:
- **Granularidad Óptima:** Tenemos una visión semanal por Municipio y Artículo, lo que permite captar patrones de demanda localizados.
- **Enriquecimiento Externo:** Hemos fusionado con éxito el calendario deportivo ciclista y las variables meteorológicas de OpenMeteo, proporcionando al modelo la "explicación" de por qué una semana pudo ser mejor o peor que otra.
- **Horizonte Temporal Sano:** Al filtrar el periodo 2019-2021, hemos eliminado el ruido atípico del COVID-19 y la falta de datos externos pasados, dejando una señal limpia de 2022 a 2024.
- **Features Temporales (Lags):** El modelo ya cuenta con "memoria" gracias a los rezagos semanales y mensuales de las unidades vendidas.

#### Próximos Pasos Sugeridos:
1. **Encoding de Variables Categóricas:** Transformar variables como `Provincia`, `Municipio` o `tipo_abc` mediante técnicas como *Target Encoding* o *Label Encoding* para que sean numéricamente interpretables.
2. **Feature Engineering Avanzada:** Crear variables de calendario (Festivos nacionales/locales, meses, trimestres).
3. **Selección de Modelo y Entrenamiento:** Dividir en Train/Test de forma cronológica (no aleatoria) y entrenar nuestro primer regresor de Gradient Boosting.
4. **Validación de Métricas:** Evaluar con RMSE y MAE la precisión de nuestra predicción de demanda.

*El dataset está listo para ser guardado y consumido en la fase de modelado.*

In [17]:
# Guardamos el dataset final de modelado
df_final.to_csv('../Datasets/df_final_modelado.csv', sep=';', index=False)
print("Dataset guardado en: '../Datasets/df_final_modelado.csv'")

Dataset guardado en: '../Datasets/df_final_modelado.csv'
